In [ ]:
%pip install fastf1

In [ ]:
import fastf1
from fastf1 import plotting
import matplotlib.pyplot as plt
from datetime import datetime
from zoneinfo import ZoneInfo
import matplotlib

aest_tz = ZoneInfo("Australia/Melbourne")

yearsToCollect = [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

In [ ]:
import logging

logging.getLogger("fastf1").setLevel(logging.CRITICAL)

Get the event

In [ ]:

calendar_2025 = fastf1.get_events_remaining(
    dt=datetime.now(),
    include_testing=True,
    backend='ergast'
)

allNextRaceData = calendar_2025.iloc[0]

nextRace = calendar_2025.iloc[0][['EventName', 'EventDate', 'Location']]

calendar_2025['EventDate'] = (
    calendar_2025['EventDate']
    .dt.tz_localize('UTC')
    .dt.tz_convert('Australia/Sydney')
)


print(f"\nNext Race: {nextRace['EventName']} in {nextRace['Location']} at {nextRace['EventDate']}")
# print(allNextRaceData)

Average lap time over past n years

In [ ]:
eventName = nextRace['EventName']
eventDate = nextRace['EventDate']
eventLocation = nextRace['Location']

fastestLapData = []
averageLapData = []

for year in yearsToCollect:
    try:
        print(f"processing {year} {eventName}")
        session = fastf1.get_session(year, nextRace['EventName'], 'R')
        
        session.load()
        
        laps = session.laps
        fastest_lap = laps.pick_fastest()
        fastest_lap_driver = fastest_lap['Driver']
        fastest_lap_time = fastest_lap['LapTime'].total_seconds() * 1000
        fastest_lap_number = fastest_lap['LapNumber']
        
        average_lap_time = laps.LapTime.mean().total_seconds() * 1000
        
        averageLapData.append((year, average_lap_time))
        fastestLapData.append((year, fastest_lap_driver, fastest_lap_time, fastest_lap_number))
        
        # print(f"{year} Fastest Lap: {fastest_lap_time} by {fastest_lap_driver} on lap {fastest_lap_number}")
    
    except Exception as e:
        print(f"Could not process {year} {eventName}: {e}")
        print("-----" * 10)
        continue
    

print("-----" * 20)
for data in fastestLapData:
    year, driver, lap_time, lap_number = data
    print(f"{year} Fastest Lap: {lap_time} by {driver} on lap {lap_number}")
    print("-----" * 20)


for data in averageLapData:
    year, avg_lap_time = data
    print(f"{year} Average Lap Time: {avg_lap_time}")
    print("-----" * 20)

In [ ]:
years = [data[0] for data in fastestLapData]
# Convert milliseconds to seconds
fastest_times = [data[2]/1000 for data in fastestLapData]
avg_times = [data[1]/1000 for data in averageLapData]

plt.figure(figsize=(12, 6))
plt.plot(years, fastest_times, 'o-', color='red', label='Fastest Lap Times')
plt.plot(years, avg_times, 'o-', color='blue', label='Average Lap Times')

# Add labels for fastest lap times
for x, y in zip(years, fastest_times):
    plt.annotate(f'{y:.3f}s', 
                (x, y),
                textcoords="offset points",
                xytext=(0,10),
                ha='center',
                color='red',
                fontsize=8)

# Add labels for average lap times
for x, y in zip(years, avg_times):
    plt.annotate(f'{y:.3f}s', 
                (x, y),
                textcoords="offset points",
                xytext=(0,-15),
                ha='center',
                color='blue',
                fontsize=8)

plt.title(f'Lap Times Evolution: {eventName} ({eventLocation})')
plt.xlabel('Year')
plt.ylabel('Lap Time (seconds)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()

# Rotate x-axis labels for better readability
plt.xticks(years, rotation=45)

# Add some padding to prevent label cutoff
plt.tight_layout()

plt.show()

To calculate the time lost on an in-lap and out-lap we can compare (DURING GREEN FLAG)

average_in_lap = average(average lap time - average in-lap)

average_out_lap = average(average lap time - average out-lap)

In [ ]:
import numpy as np

in_lap_differences = []
out_lap_differences = []

for year in yearsToCollect:
    try:
        print(f"Processing pit lap analysis for {year} {eventName}")
        # load only laps (faster, avoids telemetry downloads)
        session = fastf1.get_session(year, eventName, 'R')
        session.load(laps=True)

        laps = session.laps
        if laps is None or laps.empty:
            print(f"  No laps for {year}; skipping.")
            continue

        # get pit-related laps (may raise if not available for season)
        try:
            pit_both = laps.pick_box_laps('both')
            in_laps = laps.pick_box_laps('in')
            out_laps = laps.pick_box_laps('out')
        except Exception as e:
            print(f"  Pit data not available for {year}: {e}")
            continue

        # drop laps where they come out of pits and pit again (rare)
        racing_laps = laps.drop(pit_both.index, errors='ignore')

        def mean_secs(df):
            if df is None or df.empty:
                return np.nan
            return df['LapTime'].dropna().dt.total_seconds().mean()

        avg_normal = mean_secs(racing_laps)
        avg_in = mean_secs(in_laps)
        avg_out = mean_secs(out_laps)

        if not np.isnan(avg_in) and not np.isnan(avg_normal):
            in_lap_differences.append(avg_in - avg_normal)
        if not np.isnan(avg_out) and not np.isnan(avg_normal):
            out_lap_differences.append(avg_out - avg_normal)

    except Exception as e:
        print(f"  Could not process {year}: {e}")
        continue

avg_in_lap_loss = sum(in_lap_differences) / len(in_lap_differences) if in_lap_differences else float('nan')
avg_out_lap_loss = sum(out_lap_differences) / len(out_lap_differences) if out_lap_differences else float('nan')

print(f"Average in-lap loss:  {round(avg_in_lap_loss, 3)}")
print(f"Average out-lap loss: {round(avg_out_lap_loss, 3)}")
if not np.isnan(avg_in_lap_loss) and not np.isnan(avg_out_lap_loss):
    print(f"Total average pit loss: {round(avg_in_lap_loss + avg_out_lap_loss, 3)}")

to model car 

We have to measure how the tyres increasing in age impact the lap time

to cater for traffic we have to ensure that the car we are collecting data on is not within 10 seconds of the car in front at any point in time.

To measure the impact of a lap on tyres, we will use Q3 (qualifying) to measure the impact of a racing lap on the tyres. 

Top-end / flow = how quickly the lap is covered (long straights, high average speed, lower sustained slip)

corner / traction severity = how much time is spent cornering / braking / accelerating, lateral / longitude energy into the rubber 